# Day 3 · CUDA-Q 기반 양자 보행 (Quantum Walk)
**2026 서울퀀텀캠퍼스 양자컴퓨팅 활용과정 겨울캠프** · 12월 2일(수) · 연세대학교 박병규

| 시간 | 내용 |
|---|---|
| 10:00–11:00 | **Part 1** 양자 보행 모델 정의, 양자 상태 설정 |
| 11:00–12:00 | **Part 2** 게이트 적용, 이동(shift) 연산자 만들기 |
| 13:30–15:00 | **Part 3** 단계별 시뮬레이션, 간섭과 확률 분포 시각화 |
| 15:00–16:00 | **Part 4** 실험 조건 바꾸기, 고전 랜덤 워크와 비교 |
| 16:00–17:00 | **Part 5** 복잡한 그래프로 확장하기 |
| 보너스 | 실제 양자컴퓨터(QPU)에서 짧은 양자 보행 실행 |

> **이 노트북 사용법**  
> - 위에서부터 셀을 하나씩 실행하세요 (`Shift + Enter`).  
> - ✏️ 표시가 있는 셀은 여러분이 `___` 부분을 채우거나 숫자를 바꿔 보는 셀입니다.  
> - 🤔 표시는 실행 전에 결과를 먼저 예측해 보는 질문입니다.

In [ ]:
# 준비: 필요한 도구 불러오기
import warnings; warnings.filterwarnings("ignore")   # 경고 메시지 숨기기 (cudaq보다 먼저)
import cudaq
import numpy as np
import matplotlib.pyplot as plt

print("CUDA-Q 버전:", cudaq.__version__)
print("현재 시뮬레이터:", cudaq.get_target().name)

---
## Part 1 · 양자 보행 모델 정의와 양자 상태 설정 (10:00–11:00)

### 1-1. 먼저, 고전 랜덤 워크
원형 트랙 위에 8개의 칸(0~7)이 있고, 말 하나가 있다고 해 봅시다.

1. **동전을 던진다** → 앞면이면 오른쪽(+1), 뒷면이면 왼쪽(−1)
2. **한 칸 이동한다**
3. 반복

여러 번 반복하면 말은 출발점 근처에 가장 많이 머물고, 넓게 퍼지는 속도는 느립니다. (걸음 수 $t$에 대해 대략 $\sqrt{t}$)

### 1-2. 양자 보행 = 동전도, 말도 큐비트로
| 고전 | 양자 |
|---|---|
| 동전: 앞 **또는** 뒤 | **coin 큐비트**: $\lvert0\rangle$과 $\lvert1\rangle$의 **중첩** |
| 말의 위치: 한 칸 | **위치 레지스터**(큐비트 여러 개): 여러 칸의 **중첩** |
| 확률을 더함 | **진폭**을 더함 → 서로 **간섭**(보강/상쇄) |

8칸을 표현하려면 큐비트 몇 개가 필요할까요? $2^3 = 8$ 이므로 **위치 큐비트 3개**면 됩니다.

이 노트북에서는 큐비트를 한 줄로 이렇게 배치합니다.
```
q[0]      : coin 큐비트 (0 = 왼쪽, 1 = 오른쪽)
q[1]~q[n] : 위치 큐비트 (q[1]이 가장 낮은 자리, 즉 2진수의 1의 자리)
```
예) 위치 5 = 2진수 `101` → q[1]=1, q[2]=0, q[3]=1

### 1-3. 실습 ① 말을 원하는 칸에 놓기
처음에는 모든 큐비트가 $\lvert0\rangle$ → 말은 0번 칸에 있습니다.  
원하는 칸에 두려면, 그 칸 번호의 2진수에서 **1인 자리에 X 게이트**(0↔1 뒤집기)를 걸면 됩니다.

In [ ]:
@cudaq.kernel
def place(n_pos: int, start: int):
    q = cudaq.qvector(n_pos + 1)        # coin 1개 + 위치 n_pos개
    for b in range(n_pos):
        if (start >> b) & 1:            # start의 b번째 자리가 1이면
            x(q[b + 1])                 #   해당 위치 큐비트를 뒤집는다
    mz(q.back(n_pos))                   # 위치 큐비트만 측정

def to_position(bitstring):
    # CUDA-Q는 q[1], q[2], ... 순서로 문자를 적어 준다 → 뒤집어서 2진수로 읽기
    return int(bitstring[::-1], 2)

# ✏️ 말을 몇 번 칸에 놓을까요? (0~7)
start = ___
result = cudaq.sample(place, 3, start, shots_count=100)
for bits, count in result.items():
    print(f"측정 결과 {bits}  →  위치 {to_position(bits)}  ({count}회)")

🤔 **질문** 100번 측정했는데 왜 항상 같은 결과가 나올까요?  
→ 아직 중첩이 없기 때문입니다. 말은 정확히 한 칸에 있습니다.

### 1-4. 실습 ② 양자 동전 만들기
coin 큐비트에 **H(아다마르) 게이트**를 걸면 $\lvert0\rangle$과 $\lvert1\rangle$이 반반 섞인 중첩이 됩니다.  
H 다음에 **S 게이트**까지 걸면 $\frac{1}{\sqrt2}(\lvert0\rangle + i\lvert1\rangle)$, 즉 "허수 $i$가 붙은 동전"이 됩니다. 측정하면 똑같이 반반이지만, Part 3에서 이 차이가 **분포의 모양**을 바꾸는 것을 보게 됩니다.

In [ ]:
@cudaq.kernel
def coin_state(use_s: int):
    c = cudaq.qubit()
    h(c)
    if use_s == 1:
        s(c)

for use_s in [0, 1]:
    amp = np.array(cudaq.get_state(coin_state, use_s))
    name = "H" if use_s == 0 else "H + S"
    print(f"{name:6s} 진폭: |0>={np.round(amp[0],3)},  |1>={np.round(amp[1],3)}   "
          f"측정 확률: {np.round(np.abs(amp)**2, 3)}")

> **핵심 정리 (Part 1)**  
> - 위치 $N=2^n$칸 → 위치 큐비트 $n$개 + coin 큐비트 1개  
> - 시작 위치 = X 게이트로 설정, 동전 = H (와 S) 게이트로 중첩 만들기  
> - 측정 확률이 같아도 **진폭(위상)** 은 다를 수 있다 → 이게 간섭의 재료

---
## Part 2 · 게이트 적용과 이동(shift) 연산자 (11:00–12:00)

양자 보행의 한 걸음은 두 단계입니다.

$$\text{한 걸음} = \underbrace{\text{동전 던지기}}_{H \text{ on coin}} \;\rightarrow\; \underbrace{\text{coin에 따라 이동}}_{\text{shift 연산자 } S}$$

$$S\,\lvert1\rangle\lvert x\rangle = \lvert1\rangle\lvert x+1\rangle,\qquad S\,\lvert0\rangle\lvert x\rangle = \lvert0\rangle\lvert x-1\rangle \quad(\text{원형이므로 } \bmod N)$$

### 2-1. "+1 하기"를 게이트로 만들기
2진수에 1을 더하는 것은 **자리올림**과 같습니다.
- 가장 낮은 자리는 **항상** 뒤집힌다
- 어떤 자리는 **그보다 낮은 자리가 모두 1일 때만** 뒤집힌다

"~일 때만 뒤집기"가 바로 **제어-X(controlled-X)** 게이트입니다. 여기에 "coin이 1일 때만"이라는 조건을 하나 더 얹으면 **coin이 1일 때만 +1** 하는 회로가 됩니다.

In [ ]:
@cudaq.kernel
def increment(q: cudaq.qview):
    # q[0]=coin, q[1..n]=위치.  coin이 1이면 위치 +1
    n = q.size() - 1
    for k in range(n):
        i = n - k                     # 높은 자리부터 처리
        x.ctrl(q.front(i), q[i])      # q[0]..q[i-1]이 모두 1이면 q[i]를 뒤집기

@cudaq.kernel
def test_increment(n_pos: int, start: int):
    q = cudaq.qvector(n_pos + 1)
    x(q[0])                           # coin = 1 (오른쪽)
    for b in range(n_pos):
        if (start >> b) & 1:
            x(q[b + 1])
    increment(q)
    mz(q.back(n_pos))

print(cudaq.draw(test_increment, 3, 0))

🤔 **질문** 위치 7(`111`)에서 +1을 하면 어디로 갈까요? 실행 전에 예측해 보세요.

In [ ]:
print("시작 → +1 결과")
for start in range(8):
    r = cudaq.sample(test_increment, 3, start, shots_count=10)
    print(f"  {start}  →  {to_position(r.most_probable())}")

7 다음이 0 → 자연스럽게 **원형 트랙**이 됩니다.

### 2-2. "−1 하기"와 완성된 shift 연산자
−1은 새로 만들 필요가 없습니다. **위치 비트를 모두 뒤집고 → +1 → 다시 모두 뒤집으면** −1이 됩니다. (2진수 보수의 성질)  
"coin이 **0**일 때"를 조건으로 쓰려면 coin도 잠깐 뒤집었다가 되돌리면 됩니다.

In [ ]:
@cudaq.kernel
def shift(q: cudaq.qview):
    # coin=1 → +1
    increment(q)
    # coin=0 → -1   (coin과 위치를 뒤집고 +1 한 뒤 되돌리기)
    x(q[0])
    for i in range(1, q.size()):
        x(q[i])
    increment(q)
    for i in range(1, q.size()):
        x(q[i])
    x(q[0])

@cudaq.kernel
def qwalk(n_pos: int, start: int, steps: int, sym: int):
    q = cudaq.qvector(n_pos + 1)
    for b in range(n_pos):                 # 1) 시작 위치
        if (start >> b) & 1:
            x(q[b + 1])
    if sym == 1:                           # 2) 시작 동전: (|0> + i|1>)/√2
        h(q[0])
        s(q[0])
    for t in range(steps):                 # 3) 걸음 반복
        h(q[0])                            #    동전 던지기
        shift(q)                           #    이동

@cudaq.kernel
def qwalk_measure(n_pos: int, start: int, steps: int, sym: int):
    q = cudaq.qvector(n_pos + 1)
    for b in range(n_pos):
        if (start >> b) & 1:
            x(q[b + 1])
    if sym == 1:
        h(q[0])
        s(q[0])
    for t in range(steps):
        h(q[0])
        shift(q)
    mz(q.back(n_pos))

print(cudaq.draw(qwalk, 3, 0, 1, 0))

### 2-3. 실습 ③ 몇 걸음 걸어 보기
8칸 트랙의 **4번 칸**에서 출발해 1, 2, 3걸음을 걸어 봅시다.

In [ ]:
def show_counts(result, n_pos, title=""):
    N = 2 ** n_pos
    counts = np.zeros(N)
    for bits, c in result.items():
        counts[to_position(bits)] += c
    plt.figure(figsize=(5, 2.4))
    plt.bar(range(N), counts / counts.sum(), color="#4C72B0")
    plt.xticks(range(N)); plt.xlabel("position"); plt.ylabel("probability")
    plt.title(title); plt.tight_layout(); plt.show()

# ✏️ 걸음 수를 바꿔 보세요
for steps in [1, 2, 3]:
    r = cudaq.sample(qwalk_measure, 3, 4, steps, 0, shots_count=2000)
    show_counts(r, 3, f"8-cycle, start=4, steps={steps}")

🤔 **질문** 2걸음 뒤에는 왜 홀수 칸(3, 5)에 한 번도 나타나지 않을까요?  
→ 매 걸음 반드시 ±1 움직이므로, 짝수 걸음 뒤에는 출발점과 홀짝이 같은 칸에만 있을 수 있습니다. (고전도 마찬가지)

> **핵심 정리 (Part 2)**  
> - 한 걸음 = coin에 H → shift  
> - shift = "coin=1이면 +1, coin=0이면 −1" = 제어-X 게이트들의 조합  
> - 원형 트랙은 2진수 덧셈의 자리넘침으로 자동으로 만들어진다

---
## Part 3 · 단계별 시뮬레이션, 간섭, 분포 시각화 (13:30–15:00)

이제 트랙을 **32칸**(위치 큐비트 5개)으로 늘리고, 가운데(16번)에서 출발시킵니다.

실제 양자컴퓨터는 측정해서 **샘플**만 얻을 수 있지만, 시뮬레이터에서는 `cudaq.get_state`로 **정확한 확률 분포**를 바로 볼 수 있습니다. 이론 분포를 볼 때는 `get_state`, 실험처럼 볼 때는 `sample`을 씁니다.

In [ ]:
def walk_probs(n_pos, start, steps, sym=1):
    # 위치별 정확한 확률 (coin은 합쳐서 무시)
    state = np.array(cudaq.get_state(qwalk, n_pos, start, steps, sym))
    p = np.abs(state) ** 2
    return p.reshape(2 ** n_pos, 2).sum(axis=1)   # 인덱스 = 2*위치 + coin

N_POS, START = 5, 16
N = 2 ** N_POS
x_axis = np.arange(N) - START          # 출발점 기준 거리

fig, axes = plt.subplots(1, 4, figsize=(13, 2.6), sharey=True)
for ax, T in zip(axes, [0, 3, 8, 14]):
    ax.bar(x_axis, walk_probs(N_POS, START, T), color="#4C72B0")
    ax.set_title(f"step {T}"); ax.set_xlabel("distance from start")
axes[0].set_ylabel("probability")
plt.tight_layout(); plt.show()

### 3-1. 시간에 따른 변화를 한 장으로: 히트맵
가로축은 위치, 세로축은 걸음 수입니다. 밝을수록 확률이 높습니다.

In [ ]:
T_MAX = 15
heat = np.array([walk_probs(N_POS, START, T) for T in range(T_MAX + 1)])

plt.figure(figsize=(7, 3.6))
plt.imshow(heat, aspect="auto", origin="lower", cmap="magma",
           extent=[x_axis[0] - .5, x_axis[-1] + .5, -.5, T_MAX + .5])
plt.colorbar(label="probability")
plt.xlabel("distance from start"); plt.ylabel("step")
plt.title("Quantum walk spreading (Hadamard coin, symmetric start)")
plt.tight_layout(); plt.show()

**V자 모양**이 보이나요? 확률이 가운데에 머물지 않고 **양쪽 끝으로 빠르게 달려 나갑니다.**

### 3-2. 왜 이렇게 될까? 간섭
- 고전: "왼쪽으로 갔다가 오른쪽" 경로와 "오른쪽으로 갔다가 왼쪽" 경로의 **확률을 더함** → 항상 쌓임 → 가운데가 높아짐
- 양자: 각 경로의 **진폭을 더함**. 진폭은 +, −, 복소수가 될 수 있어서 가운데로 돌아오는 경로들은 서로 **상쇄**되고, 바깥으로 가는 경로들은 **보강**됩니다.

### 3-3. 실습 ④ 시작 동전을 바꾸면?
coin을 $\lvert0\rangle$에서 시작하는 경우(sym=0)와, Part 1에서 만든 `H+S` 동전 $\frac{1}{\sqrt2}(\lvert0\rangle+i\lvert1\rangle)$에서 시작하는 경우(sym=1)를 비교해 봅시다.

In [ ]:
# ✏️ sym 값을 0 / 1로 바꿔가며 비교
T = 14
fig, axes = plt.subplots(1, 2, figsize=(10, 2.6), sharey=True)
for ax, sym, label in zip(axes, [0, 1], ["coin start |0>", "coin start (|0>+i|1>)/√2"]):
    ax.bar(x_axis, walk_probs(N_POS, START, T, sym), color="#C44E52" if sym == 0 else "#4C72B0")
    ax.set_title(f"{label}, step {T}"); ax.set_xlabel("distance from start")
axes[0].set_ylabel("probability")
plt.tight_layout(); plt.show()

🤔 **질문** 측정하면 둘 다 50:50인 동전인데, 왜 결과가 한쪽으로 쏠릴까요?  
→ 측정 확률은 같아도 **위상**이 다르고, 위상이 간섭 패턴을 바꾸기 때문입니다. 고전 동전에는 없는 성질입니다.

> **핵심 정리 (Part 3)**  
> - 양자 보행은 가운데가 아니라 **양 끝**에 확률이 몰린다  
> - 원인은 **진폭의 간섭**  
> - 초기 coin 상태(위상)만 바꿔도 분포가 달라진다

---
## Part 4 · 실험 조건 바꾸기, 고전 랜덤 워크와 비교 (15:00–16:00)

### 4-1. 고전 랜덤 워크 계산하기
고전은 확률을 반반씩 왼쪽/오른쪽으로 나누어 주면 됩니다.

In [ ]:
def classical_probs(N, start, steps):
    p = np.zeros(N); p[start] = 1
    for _ in range(steps):
        p = 0.5 * np.roll(p, 1) + 0.5 * np.roll(p, -1)
    return p

T = 14
plt.figure(figsize=(7, 2.8))
plt.bar(x_axis - 0.2, classical_probs(N, START, T), width=0.4, label="classical", color="#999999")
plt.bar(x_axis + 0.2, walk_probs(N_POS, START, T), width=0.4, label="quantum", color="#4C72B0")
plt.xlabel("distance from start"); plt.ylabel("probability"); plt.title(f"step {T}")
plt.legend(); plt.tight_layout(); plt.show()

### 4-2. 얼마나 빨리 퍼지나? 표준편차 비교
퍼진 정도를 **표준편차 $\sigma$** 로 재 봅시다.

In [ ]:
steps_list = np.arange(0, 15)
sig_q = [np.sqrt((walk_probs(N_POS, START, T) * x_axis**2).sum()) for T in steps_list]
sig_c = [np.sqrt((classical_probs(N, START, T) * x_axis**2).sum()) for T in steps_list]

plt.figure(figsize=(5.5, 3.2))
plt.plot(steps_list, sig_c, "o-", color="#999999", label="classical  (~ √t)")
plt.plot(steps_list, sig_q, "o-", color="#4C72B0", label="quantum  (~ t)")
plt.xlabel("step t"); plt.ylabel("spread σ"); plt.legend(); plt.tight_layout(); plt.show()

print(f"14걸음 후  고전 σ = {sig_c[-1]:.2f},  양자 σ = {sig_q[-1]:.2f}")

고전은 $\sigma \approx \sqrt{t}$ (확산), 양자는 $\sigma \approx 0.54\,t$ (**탄도적 이동**).  
걸음 수가 4배가 되면 고전은 2배만 퍼지지만, 양자는 4배 퍼집니다. 이 **2차적 속도 차이**가 양자 보행 기반 탐색 알고리즘의 출발점입니다.

### 4-3. 실습 ⑤ 측정 횟수(shots)를 바꾸면?
실제 양자컴퓨터에서는 정확한 분포가 아니라 **측정 샘플**만 얻습니다. shots가 적으면 분포가 들쭉날쭉합니다.

In [ ]:
T = 10
exact = walk_probs(N_POS, START, T)

# ✏️ shots 목록을 바꿔 보세요
shots_list = [20, 200, 2000]

fig, axes = plt.subplots(1, len(shots_list), figsize=(4 * len(shots_list), 2.6), sharey=True)
for ax, shots in zip(axes, shots_list):
    r = cudaq.sample(qwalk_measure, N_POS, START, T, 1, shots_count=shots)
    est = np.zeros(N)
    for bits, c in r.items():
        est[to_position(bits)] += c / shots
    err = 0.5 * np.abs(est - exact).sum()          # 분포 차이 (total variation)
    ax.bar(x_axis, est, color="#55A868", label="sampled")
    ax.plot(x_axis, exact, "k.", label="exact")
    ax.set_title(f"shots={shots}, error={err:.2f}"); ax.set_xlabel("distance")
axes[0].legend(); plt.tight_layout(); plt.show()

### 4-4. 미니 과제
아래 표를 직접 채워 보세요. (셀을 복사해서 숫자만 바꾸면 됩니다)

| 조건 | 예측 | 실제 결과 |
|---|---|---|
| 걸음 수를 20으로 늘리면 (32칸 트랙) | | |
| 시작 coin을 sym=0으로 | | |
| 트랙을 16칸(n_pos=4)으로 줄이고 12걸음 | | |

💡 힌트: 걸음 수가 트랙 절반을 넘으면 양 끝에서 퍼진 파동이 반대편에서 **다시 만나** 간섭합니다.

In [ ]:
# ✏️ 자유 실험 셀: 값을 채워 넣으세요
n_pos, start, T, sym = 5, 16, ___, ___
p = walk_probs(n_pos, start, T, sym)
plt.figure(figsize=(6, 2.4))
plt.bar(np.arange(2 ** n_pos), p); plt.xlabel("position"); plt.ylabel("probability")
plt.title(f"n_pos={n_pos}, start={start}, steps={T}, sym={sym}"); plt.tight_layout(); plt.show()

---
## Part 5 · 복잡한 그래프로 확장하기 (16:00–17:00)

원형 트랙은 각 칸의 이웃이 2개뿐이었습니다. 이웃이 더 많은 그래프에서는 어떨까요?

### 5-1. 하이퍼큐브: 반대편 꼭짓점까지 누가 먼저 갈까?
4차원 하이퍼큐브는 꼭짓점 16개(`0000`~`1111`)이고, **비트 하나만 다른** 꼭짓점끼리 연결됩니다. 각 꼭짓점의 이웃은 4개입니다.
- 위치: 큐비트 4개
- 동전: 방향이 4개이므로 coin 큐비트 2개
- 이동: coin 값이 $j$이면 위치의 $j$번째 비트를 뒤집기
- 동전 던지기: 4면 동전에 맞는 **Grover 동전**(H로 감싼 반사)

`0000`에서 출발해서 정반대 꼭짓점 `1111`에 있을 확률을 봅시다.

In [ ]:
@cudaq.kernel
def grover_coin(c: cudaq.qview):
    h(c); x(c)
    z.ctrl(c[0], c[1])
    x(c); h(c)

@cudaq.kernel
def hyper_shift(c: cudaq.qview, p: cudaq.qview):
    for j in range(4):                    # coin 값이 j일 때만 p[j]를 뒤집기
        if (j & 1) == 0:
            x(c[0])
        if (j & 2) == 0:
            x(c[1])
        x.ctrl(c, p[j])
        if (j & 1) == 0:
            x(c[0])
        if (j & 2) == 0:
            x(c[1])

@cudaq.kernel
def hyper_walk(steps: int):
    c = cudaq.qvector(2)
    p = cudaq.qvector(4)
    h(c)                                  # 4방향 균등 중첩에서 출발
    for t in range(steps):
        grover_coin(c)
        hyper_shift(c, p)

def hyper_probs(T):
    st = np.array(cudaq.get_state(hyper_walk, T))
    return (np.abs(st) ** 2).reshape(16, 4).sum(axis=1)   # 인덱스 = coin + 4*위치

# 고전: 매 걸음 4개 비트 중 하나를 무작위로 뒤집기
Pc = np.zeros((16, 16))
for v in range(16):
    for j in range(4):
        Pc[v ^ (1 << j), v] = 0.25

Ts = np.arange(0, 13)
q_far = [hyper_probs(T)[15] for T in Ts]
c_far = [(np.linalg.matrix_power(Pc, T)[:, 0])[15] for T in Ts]

plt.figure(figsize=(5.5, 3))
plt.plot(Ts, c_far, "o-", color="#999999", label="classical")
plt.plot(Ts, q_far, "o-", color="#4C72B0", label="quantum")
plt.xlabel("step"); plt.ylabel("P(at 1111)"); plt.legend()
plt.title("4D hypercube: reaching the opposite corner"); plt.tight_layout(); plt.show()
print(f"4걸음 후 반대편 도착 확률  고전 {c_far[4]:.3f}  vs  양자 {q_far[4]:.3f}")

양자 보행은 **딱 4걸음**(최단 거리) 만에 반대편에 56% 확률로 도착하지만, 고전은 아무리 걸어도 약 12% 근처를 넘지 못합니다. 하이퍼큐브 위의 양자 보행은 실제로 **양자 탐색 알고리즘**(SKW 알고리즘)의 기반이 됩니다.

### 5-2. 임의의 네트워크: 연속 시간 양자 보행
coin 없이도 양자 보행을 정의할 수 있습니다. 그래프의 연결 행렬 $A$로
$$\lvert\psi(t)\rangle = e^{-iAt}\lvert\text{start}\rangle$$
이것을 **연속 시간 양자 보행(CTQW)** 이라고 합니다.

생물학·사회 네트워크에서 많이 쓰는 **네트워크 전파(network propagation)**, 즉 열이 퍼지듯 $e^{-Lt}$로 신호를 퍼뜨리는 방법의 "양자 버전"이라고 볼 수 있습니다.

아래는 다리 노드로 이어진 **두 개의 모듈**로 이루어진 8노드 네트워크입니다. (예: 두 기능 모듈 사이를 잇는 유전자)

In [ ]:
import networkx as nx
from scipy.linalg import expm

# ✏️ 연결(엣지)을 바꿔서 나만의 네트워크를 만들어 보세요 (노드 0~7)
edges = [(0, 1), (0, 2), (1, 2), (2, 3),      # 모듈 A
         (3, 4),                              # 다리
         (4, 5), (4, 6), (5, 6), (6, 7), (5, 7)]  # 모듈 B

G = nx.Graph(); G.add_nodes_from(range(8)); G.add_edges_from(edges)
A = nx.to_numpy_array(G, nodelist=range(8))

plt.figure(figsize=(4.5, 2.6))
nx.draw(G, pos=nx.spring_layout(G, seed=3), with_labels=True,
        node_color=["#DD8452"] * 4 + ["#4C72B0"] * 4, node_size=450, font_color="white")
plt.title("8-node network"); plt.show()

$e^{-iA\,dt}$는 8×8 유니터리 행렬이고, 큐비트 3개에 걸 수 있는 **사용자 정의 게이트**로 등록할 수 있습니다. 이것을 여러 번 반복하면 시간이 흐릅니다.

(참고: CUDA-Q의 사용자 정의 게이트는 q[0]을 가장 높은 자리로 읽기 때문에, 비트 순서를 맞춰 주는 변환 `P`를 한 번 곱해 줍니다.)

In [ ]:
DT = 0.25
rev = [int(f"{v:03b}"[::-1], 2) for v in range(8)]    # 비트 순서 맞추기
P = np.eye(8)[rev]
U = P @ expm(-1j * A * DT) @ P.T
cudaq.register_operation("ctqw_step", U.flatten())

@cudaq.kernel
def ctqw(start: int, k: int):
    q = cudaq.qvector(3)
    for b in range(3):
        if (start >> b) & 1:
            x(q[b])
    for i in range(k):
        ctqw_step(q[0], q[1], q[2])

def ctqw_probs(start, k):
    return np.abs(np.array(cudaq.get_state(ctqw, start, k))) ** 2

# 정확한 해와 비교 (검증)
e0 = np.zeros(8); e0[0] = 1
print("CUDA-Q 회로와 정확한 해가 일치하는가?",
      np.allclose(ctqw_probs(0, 8), np.abs(expm(-1j * A * 8 * DT) @ e0) ** 2))

In [ ]:
# 노드 0에서 출발: 양자 보행 vs 고전 확산(열 방정식 e^{-Lt})
L = np.diag(A.sum(1)) - A
K = 24
qt = np.array([ctqw_probs(0, k) for k in range(K + 1)])
ct = np.array([expm(-L * k * DT) @ e0 for k in range(K + 1)])

fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
for ax, data, title in zip(axes, [ct, qt], ["classical diffusion", "quantum walk (CTQW)"]):
    im = ax.imshow(data, aspect="auto", origin="lower", cmap="magma", vmin=0, vmax=0.6,
                   extent=[-.5, 7.5, 0, K * DT])
    ax.axvline(3.5, color="white", ls="--", lw=1)
    ax.set_xlabel("node"); ax.set_title(title)
axes[0].set_ylabel("time")
fig.colorbar(im, ax=axes, label="probability"); plt.show()

print("모듈 B(노드 4~7)로 넘어간 확률의 최댓값")
print(f"  고전 확산: {ct[:, 4:].sum(1).max():.2f}   양자 보행: {qt[:, 4:].sum(1).max():.2f}")

🤔 **토론**
- 고전 확산은 천천히 고르게 퍼져서 결국 모든 노드가 비슷해집니다.
- 양자 보행은 **진동**하면서 특정 노드에 확률이 다시 몰리기도 합니다. 네트워크의 **구조(대칭성, 다리)** 에 민감하게 반응합니다.
- 이 네트워크에서는 양자 보행이 모듈 B로 **더 적게** 넘어갑니다. 다리 노드에서 파동이 반사되고 간섭해서 모듈 A에 머무르기 때문입니다. 즉 양자 보행이 **항상 더 빠른 것은 아니며**, 그래프 구조에 따라 결과가 크게 달라집니다.
- 이렇게 구조에 민감한 성질 덕분에 그래프 비교, 중요 노드 찾기, 네트워크 기반 탐색 등에 활용되고 있습니다.

✏️ 위의 `edges`를 바꿔서 다리를 2개로 늘리거나, 모듈 사이를 끊어 보세요. 결과가 어떻게 달라지나요?

> **핵심 정리 (Part 5)**  
> - coin 방식은 이웃 수만큼 coin 상태를 늘리면 어떤 그래프로도 확장된다  
> - 하이퍼큐브: 양자는 최단 거리 만에 반대편 도착 (고전은 거의 못 감)  
> - 연속 시간 양자 보행: 네트워크 행렬 $A$만 있으면 바로 정의되며, 고전 네트워크 전파와 비교할 수 있다

---
## 보너스 · 실제 양자컴퓨터에서 양자 보행 실행하기

실제 장비에서는 게이트가 많을수록 오류가 쌓입니다. 그래서 **가장 작은 양자 보행**을 사용합니다.
- 4칸 원형 트랙 (위치 큐비트 2개 + coin 1개 = **3큐비트**)
- **2걸음**, 출발점 0

이상적인 결과: 짝수 걸음이므로 **0번과 2번 칸에만** 나타나야 합니다. 1, 3번 칸이 나온다면 그것은 **장비의 노이즈**입니다.

In [ ]:
@cudaq.kernel
def small_walk(steps: int):
    q = cudaq.qvector(3)                   # q[0]=coin, q[1..2]=위치
    for t in range(steps):
        h(q[0])
        increment(q)                       # coin=1 → +1
        x(q[0]); x(q[1]); x(q[2])
        increment(q)                       # coin=0 → -1
        x(q[1]); x(q[2]); x(q[0])
    mz(q.back(2))

print(cudaq.draw(small_walk, 2))
r_ideal = cudaq.sample(small_walk, 2, shots_count=1000)
print("이상적 시뮬레이터:", {to_position(b): c for b, c in r_ideal.items()})

### 실제 QPU 실행
> ⚠️ 실행 전 강사의 안내에 따라 QuREKA에서 사용할 **백엔드와 shots(100~1,000)** 를 확인하세요. 실제 장비 실행에는 비용(크레딧)이 듭니다. **Job ID와 shots를 꼭 기록**하세요.

아래 셀의 `set_target` 줄은 QuREKA에서 안내하는 설정으로 바꿔서 사용합니다.

In [ ]:
RUN_ON_QPU = False      # ✏️ 강사 안내 후 True로 바꾸기

if RUN_ON_QPU:
    # 예시 (QuREKA 안내에 맞게 수정): cudaq.set_target("ionq", qpu="qpu.forte-1")
    r_qpu = cudaq.sample(small_walk, 2, shots_count=100)
    print("실제 QPU:", {to_position(b): c for b, c in r_qpu.items()})
else:
    print("QPU 실행 건너뜀: 아래 노이즈 시뮬레이션으로 미리 살펴보세요.")

### QPU를 쓸 수 없을 때: 노이즈 시뮬레이션
게이트마다 약간의 무작위 오류(탈분극 노이즈)를 넣어 실제 장비를 흉내 냅니다.

In [ ]:
cudaq.set_target("density-matrix-cpu")
noise = cudaq.NoiseModel()

# ✏️ 오류율을 바꿔 보세요 (0.0 ~ 0.1)
error_rate = 0.03
noise.add_all_qubit_channel("h", cudaq.DepolarizationChannel(error_rate))
noise.add_all_qubit_channel("x", cudaq.DepolarizationChannel(error_rate))

r_noisy = cudaq.sample(small_walk, 2, shots_count=1000, noise_model=noise)
noisy = {to_position(b): c for b, c in r_noisy.items()}
print("노이즈 시뮬레이션:", dict(sorted(noisy.items())))
print(f"나오면 안 되는 홀수 칸(1, 3) 비율: {(noisy.get(1, 0) + noisy.get(3, 0)) / 1000:.1%}")

cudaq.reset_target()      # 원래 시뮬레이터로 되돌리기

---
## 보너스 2 · (소개) 다른 양자컴퓨터로 가져가기: CUDA-Q → OpenQASM → Qiskit

> 📌 강의 시간에는 강사가 화면으로 **소개**합니다. 아래 셀은 실제 양자컴퓨터에 접속하지 않고 내 컴퓨터 안에서만 실행되므로, 궁금하면 직접 돌려 봐도 됩니다. (Qiskit 필요)

양자 회로에는 장비와 상관없이 주고받을 수 있는 공용 텍스트 형식이 있습니다. 바로 **OpenQASM**입니다.
CUDA-Q로 만든 회로를 OpenQASM으로 내보내면, **Qiskit**(IBM 양자컴퓨터의 표준 도구)에서 그대로 불러올 수 있습니다.

```
CUDA-Q 커널  →  OpenQASM 2 (텍스트)  →  Qiskit QuantumCircuit  →  IBM 등 다른 장비
```

내보내려면 커널이 두 가지 조건을 지켜야 합니다.
- **인자가 없을 것**: 걸음 수 같은 값을 커널 안에 고정
- **다른 커널을 부르지 않을 것**: `increment` 대신 게이트를 직접 풀어서 쓰기

아래는 보너스 1의 `small_walk`(4칸, 2걸음)를 이 조건에 맞게 다시 쓴 것입니다. 하는 일은 똑같습니다.

In [ ]:
@cudaq.kernel
def walk2_export():
    q = cudaq.qvector(3)                   # q[0]=coin, q[1..2]=위치
    for t in range(2):                     # 2걸음 (고정)
        h(q[0])
        x.ctrl(q[0], q[1], q[2]); x.ctrl(q[0], q[1])      # coin=1 → +1
        x(q[0]); x(q[1]); x(q[2])
        x.ctrl(q[0], q[1], q[2]); x.ctrl(q[0], q[1])      # coin=0 → -1
        x(q[1]); x(q[2]); x(q[0])
    mz(q[1]); mz(q[2])

qasm = cudaq.translate(walk2_export, format="openqasm2")
print(f"OpenQASM 텍스트 {len(qasm.splitlines())}줄 중 앞부분:\n")
print("\n".join(qasm.splitlines()[:12]))

In [ ]:
try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    HAS_QISKIT = True
except ImportError:
    HAS_QISKIT = False
    print("Qiskit이 없습니다. 설치: pip install qiskit")

if HAS_QISKIT:
    qc = QuantumCircuit.from_qasm_str(qasm)
    print("Qiskit으로 불러온 회로의 게이트:", dict(qc.count_ops()))

    # 같은 결과가 나오는지 확인 (Qiskit은 비트를 q2 q1 q0 순서로 표시)
    probs = Statevector(qc.remove_final_measurements(inplace=False)).probabilities_dict()
    pos = {}
    for bits, p in probs.items():
        q2, q1 = int(bits[0]), int(bits[1])
        pos[q1 + 2 * q2] = pos.get(q1 + 2 * q2, 0) + p
    print("Qiskit 계산 위치 확률:", {k: round(float(v), 3) for k, v in sorted(pos.items())})
    print("CUDA-Q 결과(보너스 1)  :", {to_position(b): c for b, c in r_ideal.items()}, "(1000회 측정)")

### 장비 구조에 따라 회로가 달라진다
실제 장비는 아무 큐비트끼리나 2큐비트 게이트를 걸 수 없습니다. 연결되지 않은 큐비트 사이에는 **SWAP**이 추가되어 회로가 길어지고, 그만큼 **노이즈**도 늘어납니다.

같은 회로를 세 가지 연결 구조에 맞춰 변환(transpile)해 봅시다. 실제 장비의 보정 정보가 아니라 **연결 모양만 흉내 낸** 비교입니다.

In [ ]:
if HAS_QISKIT:
    from qiskit import transpile
    from qiskit.transpiler import CouplingMap

    layouts = {
        "all-to-all (이온트랩, IonQ 유형)": (None,                           ["rz", "sx", "x", "cz"]),
        "square grid (초전도, IQM 유형)":    (CouplingMap.from_grid(3, 3),    ["r", "cz"]),
        "heavy-hex (초전도, IBM 유형)":      (CouplingMap.from_heavy_hex(3),  ["rz", "sx", "x", "ecr"]),
    }
    names, twoq = [], []
    for name, (cmap, basis) in layouts.items():
        t = transpile(qc, coupling_map=cmap, basis_gates=basis, optimization_level=1, seed_transpiler=1)
        n2 = sum(v for k, v in t.count_ops().items() if k in ("cz", "ecr", "cx"))
        names.append(name.split(" (")[0]); twoq.append(n2)
        print(f"{name:32s} 2큐비트 게이트 {n2:3d}개,  깊이 {t.depth()}")

    plt.figure(figsize=(5, 2.6))
    plt.bar(names, twoq, color=["#55A868", "#4C72B0", "#C44E52"])
    plt.ylabel("2-qubit gates"); plt.title("Same circuit, different hardware layout")
    plt.tight_layout(); plt.show()

🤔 **질문** 모든 큐비트가 연결된 장비에서는 2큐비트 게이트가 28개인데, 격자 구조에서는 왜 늘어날까요? 이 차이가 측정 결과의 "나오면 안 되는 홀수 칸" 비율에 어떤 영향을 줄까요?

### IBM 양자컴퓨터에서 실행하려면 (참고)
IBM Quantum Platform에 무료 계정(Open Plan)을 만들면 실제 IBM 양자컴퓨터를 한 달에 몇 분 정도 무료로 쓸 수 있습니다. 무료 사용 시간과 조건은 바뀔 수 있으니 가입할 때 IBM 안내를 확인하세요. 계정이 있으면 아래처럼 실행합니다. (이 노트북에서는 실행하지 않습니다)

```python
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
service = QiskitRuntimeService()                 # 저장된 IBM 계정 정보 사용
backend = service.backend("백엔드_이름")
isa = transpile(qc, backend=backend, optimization_level=1)
job = SamplerV2(backend).run([isa], shots=1000)
print(job.job_id())
```

> **핵심 정리 (보너스 2)**  
> - OpenQASM은 장비와 무관한 회로 공용 형식 → CUDA-Q 회로를 Qiskit/IBM 등으로 옮길 수 있다  
> - 장비의 **연결 구조**에 따라 같은 회로도 길이가 달라지고, 노이즈도 달라진다

---
## 오늘의 정리
1. 양자 보행 = **양자 동전(coin) + 조건부 이동(shift)** 의 반복
2. shift 연산자는 **제어-X 게이트**로 만든 2진수 덧셈기
3. 진폭의 **간섭** 때문에 양자 보행은 고전보다 **빠르게(∝ t)** 퍼진다
4. 하이퍼큐브, 임의의 네트워크로 확장 가능하며 **탐색·네트워크 분석**에 활용된다
5. 실제 장비에서는 **노이즈**가 이상적인 패턴을 흐린다
6. 회로는 **OpenQASM**으로 다른 장비(IBM 등)에 옮길 수 있고, 장비의 **연결 구조**가 회로 길이를 바꾼다

### 내일 (Day 4) 예고: Quantum-HPC 하이브리드 AI
오늘은 정해진 회로를 돌렸다면, 내일은 **매개변수가 있는 회로**를 GPU와 함께 학습시켜 AI 모델처럼 사용해 봅니다.